# Uncertainty Quantification Head

Fit a two-layer Transformer head to lookback ratios and recent attention weights. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2505.08200v1)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

The estimator trains for 10 epochs by default. Each epoch reports token-weighted training BCE, validation BCE, and validation token PR-AUC and ROC-AUC. Held-out metrics use the final epoch.

In [1]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import logging

import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS, PILOT_LIMITS
from src.evaluation import TraceStore, evaluate
from src.methods import UncertaintyQuantificationHead

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("src.methods").setLevel(logging.INFO)

epochs = UncertaintyQuantificationHead.default_epochs
pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
history = []
for benchmark in DATASETS:
    print(f"Training on {benchmark}", flush=True)
    splits = store.load_splits(benchmark, limits)
    estimator = UncertaintyQuantificationHead(epochs=epochs)
    results.extend(
        {"benchmark": benchmark, **row} for row in evaluate(estimator, splits)
    )
    history.extend(
        {"benchmark": benchmark, **row} for row in estimator.training_history
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

Training on mbpp


UncertaintyQuantificationHead epoch 1/10 train_loss=0.5838 validation_loss=0.4586 validation_token_pr_auc=0.2569 validation_token_roc_auc=0.7063


UncertaintyQuantificationHead epoch 2/10 train_loss=0.5369 validation_loss=0.4374 validation_token_pr_auc=0.2303 validation_token_roc_auc=0.6721


UncertaintyQuantificationHead epoch 3/10 train_loss=0.5298 validation_loss=0.4541 validation_token_pr_auc=0.2269 validation_token_roc_auc=0.6663


UncertaintyQuantificationHead epoch 4/10 train_loss=0.5296 validation_loss=0.4478 validation_token_pr_auc=0.2499 validation_token_roc_auc=0.6905


UncertaintyQuantificationHead epoch 5/10 train_loss=0.5364 validation_loss=0.4808 validation_token_pr_auc=0.2599 validation_token_roc_auc=0.7005


... (35 outputs left out)

UncertaintyQuantificationHead epoch 8/10 train_loss=0.5328 validation_loss=0.4787 validation_token_pr_auc=0.7878 validation_token_roc_auc=0.8410


UncertaintyQuantificationHead epoch 9/10 train_loss=0.4403 validation_loss=0.5380 validation_token_pr_auc=0.7797 validation_token_roc_auc=0.8613


UncertaintyQuantificationHead epoch 10/10 train_loss=0.5623 validation_loss=0.7199 validation_token_pr_auc=0.8070 validation_token_roc_auc=0.8568


n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.4231   
               line     509         54         0.1061  0.4523   
               sample    74         31         0.4189  0.6088   
humaneval      token   3714        665         0.1791  0.2376   
               line     484         55         0.1136  0.2556   
               sample    46         19         0.4130  0.4693   
humaneval_java token   1157        205         0.1772  0.3311   
               line     155         15         0.0968  0.4464   
               sample    36         11         0.3056  0.5473   
defects4j      token   3901       2220         0.5691  0.9296   
               line     448        148         0.3304  0.8709   
               sample    86         35         0.4070  0.6545   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.4238   0.7705     0.3502  0.6076  0.4443   
               line               0.4579   0.8705     0.3571  0.5556  0.4348   
               sample             0.6169   0.6429     0.4500  0.8710  0.5934   
humaneval      token              0.2385   0.6286     0.2120  0.9594  0.3472   
               line               0.2615   0.7234     0.1536  0.9273  0.2636   
               sample             0.4834   0.4951     0.4130  1.0000  0.5846   
humaneval_java token              0.3334   0.6883     0.3464  0.2585  0.2961   
               line               0.4612   0.8781     0.3529  0.4000  0.3750   
               sample             0.5654   0.7055     0.3846  0.9091  0.5405   
defects4j      token              0.9296   0.9148     0.8338  0.8793  0.8560   
               line               0.8713   0.9218     0.7667  0.7770  0.7718   
               sample             0.6594   0.6678     0.4146  0.9714  0.5812   

                       threshold  
benchmark      level              
mbpp           token      0.5610  
               line       0.5996  
               sample     0.5996  
humaneval      token      0.1326  
               line       0.1332  
               sample     0.1343  
humaneval_java token      0.1847  
               line       0.1861  
               sample     0.1856  
defects4j      token      0.1327  
               line       0.1412  
               sample     0.1294

### Training history

Loss is binary cross-entropy on labeled tokens.

In [4]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(history).set_index(["benchmark", "epoch"]).round(4))

train_loss  validation_loss  validation_token_pr_auc  \
benchmark      epoch                                                         
mbpp           1          0.5838           0.4586                   0.2569   
               2          0.5369           0.4374                   0.2303   
               3          0.5298           0.4541                   0.2269   
               4          0.5296           0.4478                   0.2499   
               5          0.5364           0.4808                   0.2599   
               6          0.5360           0.4493                   0.3092   
               7          0.5233           0.4299                   0.3168   
               8          0.5173           0.4141                   0.3327   
               9          0.4892           0.3915                   0.3718   
               10         0.4711           0.4931                   0.4546   
humaneval      1          0.4612           0.5619                   0.2441   
               2          0.4210           0.5879                   0.2578   
               3          0.4051           0.6250                   0.2523   
               4          0.4043           0.5785                   0.2499   
               5          0.4088           0.5969                   0.2473   
               6          0.4154           0.6192                   0.2499   
               7          0.4145           0.6086                   0.2526   
               8          0.4070           0.6127                   0.2583   
               9          0.4080           0.5765                   0.2618   
               10         0.4162           0.6070                   0.2650   
humaneval_java 1          0.5118           0.3554                   0.1524   
               2          0.4749           0.3786                   0.1604   
               3          0.5120           0.3501                   0.1943   
               4          0.4631           0.3859                   0.1862   
               5          0.4864           0.3468                   0.1724   
               6          0.4851           0.3758                   0.1716   
               7          0.4683           0.3558                   0.1865   
               8          0.4658           0.3593                   0.1853   
               9          0.4679           0.3555                   0.1866   
               10         0.4667           0.3642                   0.1865   
defects4j      1          0.7362           0.6809                   0.6386   
               2          0.6931           0.6789                   0.6143   
               3          0.7162           0.6528                   0.6210   
               4          0.7090           0.6528                   0.6203   
               5          0.7174           0.6443                   0.6357   
               6          0.6519           0.7165                   0.6843   
               7          0.6734           0.6246                   0.6860   
               8          0.5328           0.4787                   0.7878   
               9          0.4403           0.5380                   0.7797   
               10         0.5623           0.7199                   0.8070   

                      validation_token_roc_auc  
benchmark      epoch                            
mbpp           1                        0.7063  
               2                        0.6721  
               3                        0.6663  
               4                        0.6905  
               5                        0.7005  
               6                        0.7397  
               7                        0.7442  
               8                        0.7463  
               9                        0.7598  
               10                       0.8458  
humaneval      1                        0.4980  
               2                        0.4387  
               3                        0.4624  
          